# NB07 — Cross-Source Robustness (standalone, Q1)

**Self-contained — builds its own splits, does not read any pre-built `source_holdout_*` file.**
The only external input is the master deduplicated dataset
(`data/processed/benchmark_cleaned.csv`, one row per comment with `uid`, `label5`, `source`, `script`,
`text_clean`). Everything else — the Bangla-only pool, the three held-out splits, the stratified
train/val partition — is constructed inside this notebook so there is no dependency on how an
earlier notebook happened to build `data/splits/source_holdout_*.csv`.

**Why that matters here.** The original `source_holdout_*` files in `data/splits/` were built under
the *first* protocol — train on the other **three** sources, test on the fourth — so for the
`facebook_44001` holdout that file's training pool is `banth + multilabel_12557 + bd_shs`, which
includes the Romanized source. Reading those files for the new protocol would silently retrain on
Banglish data in every configuration. This notebook avoids that by never touching those files: it
filters the master dataset down to the three Bangla-script sources first, so `banth` is excluded
before any split is drawn, not filtered out after the fact.

Protocol, for each of the three Bangla-script sources in turn:
- pool = the master dataset restricted to `source != banth` and `source != <held-out source>`,
  i.e. the **other two Bangla-script sources only**,
- pool is split 87.5/12.5, stratified on `label5`, into train/val,
- test = every row of the held-out source (untouched, no sampling),
- a hard `uid` intersection assertion runs before training starts,
- the proposed script-aware CE+FGM ensemble (BanglishBERT, BanglaBERT specialist, MuRIL, XLM-R)
  trains for **3 seeds**, so the reported Macro-F1 carries a standard deviation.



In [1]:
# ── Imports, device, paths ───────────────────────────────────────────────────
import os, json, time, math, random, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef, roc_auc_score
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(), "| device:", device)

ROOT      = os.environ.get("PROJECT_ROOT", "..")
# NB07 reads the single master deduplicated dataset (see cell 2 / MASTER_DATASET env var) and builds
# every split itself. It intentionally does NOT read data/splits/source_holdout_*.csv — those files
# were built for a different protocol (train on the other three sources, banth included) and would
# reintroduce the Romanized source into training if used here.
OUT       = f"{ROOT}/outputs/robustness"
PAPER     = f"{ROOT}/outputs/paper_q1"
FIG       = f"{PAPER}/figures"
TAB       = f"{PAPER}/tables"
for d in (OUT, FIG, TAB):
    os.makedirs(d, exist_ok=True)

COL = {"blue":"#2f6fb0","green":"#1a9c6e","orange":"#e8930c","grey":"#8a8f98","text":"#1e2838","muted":"#6a7280"}
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False,"figure.dpi":120})
print("assets ->", FIG, "and", TAB)


CUDA: True | device: cuda
assets -> ../outputs/paper_q1/figures and ../outputs/paper_q1/tables


In [2]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
DEBUG = False; DEBUG_SAMPLES = 400; DEBUG_EPOCHS = 2

CFG = {"models": {"banglishbert":"csebuetnlp/banglishbert",
                  "banglabert":"csebuetnlp/banglabert",
                  "muril":"google/muril-base-cased",
                  "xlmr":"xlm-roberta-base"},
       "max_length":128,"batch_size":32,"eval_batch_size":128,"grad_accum_steps":1,"num_workers":2,
       "epochs":8,"encoder_lr":2e-5,"head_lr":8e-5,"weight_decay":0.01,"warmup_ratio":0.10,
       "max_grad_norm":1.0,"label_smoothing":0.03,"focal_gamma":2.0,"class_weight_beta":0.999,
       "dropout":0.25,"head_hidden_dim":384,"n_msd":1,"patience":2,"use_fp16":torch.cuda.is_available(),
       # proposed recipe = CE + FGM only
       "use_focal":False,"use_cw":False,"use_balanced_sampler":False,"sampler_alpha":0.5,
       "use_fgm":True,"fgm_epsilon":1.0,"use_rdrop":False,"rdrop_alpha":0.5,"use_ema":False,"ema_decay":0.999,
       "banglabert_script_only":True,"banglabert_script_key":"banglabert"}

SEEDS       = [42, 123, 456]                                        # 3 seeds -> mean ± std
RUN_MODELS  = ["banglishbert","banglabert","muril","xlmr"]          # 4 encoders
LABEL, TEXT = "label5", "text_clean"
FORCE       = False
VAL_FRAC    = 0.125                                                  # 87.5/12.5 train/val on the pool
SPLIT_SEED  = 42
if DEBUG: CFG["epochs"] = DEBUG_EPOCHS; print("DEBUG MODE")

# ── Build the Bangla-only pool and the three held-out splits from scratch ──────
# IMPORTANT: this does NOT read data/splits/source_holdout_*.csv. Those files were built under the
# earlier "train on the other three sources" protocol, so for e.g. the facebook_44001 holdout, that
# file's training pool is banth + multilabel_12557 + bd_shs -- the Romanized source leaks straight in.
# Instead we go back to the single master dataset and filter out banth before any split is drawn.
MASTER_PATH = os.environ.get("MASTER_DATASET",
                              f"{ROOT}/data/processed/benchmark_cleaned.csv")
master = pd.read_csv(MASTER_PATH, encoding="utf-8-sig")
assert {LABEL, "source", "script", TEXT}.issubset(master.columns), \
    f"master dataset missing required columns; found {list(master.columns)}"
if "uid" not in master.columns:
    master["uid"] = master.index    # same convention as the original split-construction notebook
    print("uid column not found in the master CSV — assigned uid = row index (matches original convention)")

BANGLA_SOURCES = ["facebook_44001", "multilabel_12557", "bd_shs"]   # banth excluded -- never loaded below
present = set(master["source"].unique())
missing_src = [s for s in BANGLA_SOURCES if s not in present]
assert not missing_src, f"expected Bangla sources not found in master dataset: {missing_src}"

bangla_pool_all = master[master["source"].isin(BANGLA_SOURCES)].reset_index(drop=True)
assert "banth" not in set(bangla_pool_all["source"]), "banth leaked into the Bangla pool -- aborting"
print(f"Bangla-only pool: {len(bangla_pool_all):,} rows across {BANGLA_SOURCES} "
      f"(banth excluded, {len(master) - len(bangla_pool_all):,} rows dropped)")

from sklearn.model_selection import train_test_split

def build_holdout(held_source, seed=SPLIT_SEED, val_frac=VAL_FRAC):
    """Test = every row of held_source. Train/val = stratified split of the OTHER TWO Bangla
    sources only. banth never enters this function's input (bangla_pool_all already excludes it)."""
    test_df  = bangla_pool_all[bangla_pool_all["source"] == held_source].reset_index(drop=True)
    pool_df  = bangla_pool_all[bangla_pool_all["source"] != held_source].reset_index(drop=True)
    assert set(pool_df["source"].unique()) == set(BANGLA_SOURCES) - {held_source}, \
        f"train pool for {held_source} holdout contains unexpected sources: {pool_df['source'].unique()}"
    train_df, val_df = train_test_split(pool_df, test_size=val_frac, random_state=seed,
                                        stratify=pool_df[LABEL])
    seen = set(train_df["uid"]) | set(val_df["uid"])
    leaked = len(seen & set(test_df["uid"]))
    assert leaked == 0, f"LEAK in {held_source} holdout: {leaked} uids shared with test"
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

configs = [f"source_holdout_{s}" for s in BANGLA_SOURCES]
HOLDOUT_SPLITS = {}   # tag -> (train_df, val_df, test_df), built once, reused below
for tag, src in zip(configs, BANGLA_SOURCES):
    tr, va, te = build_holdout(src)
    HOLDOUT_SPLITS[tag] = (tr, va, te)
    print(f"  {tag:38s} train={len(tr):,} val={len(va):,} test={len(te):,} "
          f"| train sources={sorted(tr['source'].unique())} | leak=0 OK")

print(f"\nrobustness configs (Bangla source-held-out only, freshly built): {configs}")

uid column not found in the master CSV — assigned uid = row index (matches original convention)
Bangla-only pool: 56,989 rows across ['facebook_44001', 'multilabel_12557', 'bd_shs'] (banth excluded, 37,334 rows dropped)
  source_holdout_facebook_44001          train=12,172 val=1,739 test=43,078 | train sources=['bd_shs', 'multilabel_12557'] | leak=0 OK
  source_holdout_multilabel_12557        train=42,093 val=6,014 test=8,882 | train sources=['bd_shs', 'facebook_44001'] | leak=0 OK
  source_holdout_bd_shs                  train=45,465 val=6,495 test=5,029 | train sources=['facebook_44001', 'multilabel_12557'] | leak=0 OK

robustness configs (Bangla source-held-out only, freshly built): ['source_holdout_facebook_44001', 'source_holdout_multilabel_12557', 'source_holdout_bd_shs']


In [3]:
# ── Persist the freshly built splits (Bangla-only, banth excluded) to disk ─────
# Saved under a distinct folder name so these are never confused with the old
# data/splits/source_holdout_*.csv files, which were built under the earlier
# "train on the other three sources" protocol and include banth.
SAVE_DIR = f"{ROOT}/data/splits/source_holdout_bangla_only"
os.makedirs(SAVE_DIR, exist_ok=True)

manifest = {"protocol": "3 Bangla-script sources; banth entirely excluded from the pool",
            "bangla_sources": BANGLA_SOURCES, "val_frac": VAL_FRAC, "split_seed": SPLIT_SEED,
            "configs": {}}

for tag, (tr, va, te) in HOLDOUT_SPLITS.items():
    tr.to_csv(f"{SAVE_DIR}/{tag}_train.csv", index=False, encoding="utf-8-sig")
    va.to_csv(f"{SAVE_DIR}/{tag}_val.csv",   index=False, encoding="utf-8-sig")
    te.to_csv(f"{SAVE_DIR}/{tag}_test.csv",  index=False, encoding="utf-8-sig")
    manifest["configs"][tag] = {
        "train_n": len(tr), "val_n": len(va), "test_n": len(te),
        "train_sources": sorted(tr["source"].unique().tolist()),
        "held_out_source": tag.replace("source_holdout_", ""),
        "banth_in_training": "banth" in set(tr["source"]) | set(va["source"]),   # must be False
    }
    print(f"  saved {tag}: train={len(tr):,} val={len(va):,} test={len(te):,} -> {SAVE_DIR}/")

with open(f"{SAVE_DIR}/split_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

assert all(not c["banth_in_training"] for c in manifest["configs"].values()), \
    "manifest shows banth in a training pool -- do not proceed"
print(f"\nsplit_manifest.json written -- banth_in_training is False for all configs: verified")
print(f"all splits saved to {SAVE_DIR}/")

  saved source_holdout_facebook_44001: train=12,172 val=1,739 test=43,078 -> ../data/splits/source_holdout_bangla_only/
  saved source_holdout_multilabel_12557: train=42,093 val=6,014 test=8,882 -> ../data/splits/source_holdout_bangla_only/
  saved source_holdout_bd_shs: train=45,465 val=6,495 test=5,029 -> ../data/splits/source_holdout_bangla_only/

split_manifest.json written -- banth_in_training is False for all configs: verified
all splits saved to ../data/splits/source_holdout_bangla_only/


In [4]:
# ── Training machinery (self-contained; mirrors the main training pipeline) ──
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc, label=None):
        self.t = df.reset_index(drop=True)[TEXT].fillna("").astype(str).tolist()
        self.y = [int(enc.get(v, -1)) for v in df[label if label is not None else LABEL]]
        self.tok, self.ml = tok, maxlen
    def __len__(self): return len(self.t)
    def __getitem__(self, i):
        e = self.tok(self.t[i], max_length=self.ml, truncation=True, padding=False)
        it = {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}
        it["label"] = torch.tensor(self.y[i], dtype=torch.long)
        return it

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        lb = torch.stack([f["label"] for f in fs])
        ft = [{k: v for k, v in f.items() if k != "label"} for f in fs]
        b = self.tok.pad(ft, padding=True, return_tensors="pt"); b["label"] = lb; return b

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, ls=0.0):
        super().__init__(); self.g, self.w, self.ls = gamma, weight, ls
    def forward(self, lg, t):
        ce = F.cross_entropy(lg, t, weight=self.w, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.g) * ce).mean()

def build_cw(s, enc, beta=0.999, mw=10.0):
    m = s.map(enc).dropna().astype(int); n = len(enc); c = m.value_counts().sort_index()
    w = np.ones(n, dtype=np.float32)
    for i in range(n):
        k = int(c.get(i, 0))
        if k > 0: w[i] = 1.0 / max((1.0 - beta ** k) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * mw); w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_sampler(df, enc, label, alpha=0.5):
    y = df[label].map(enc).fillna(-1).astype(int).values
    cc = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); cc[cc == 0] = 1.0
    cw = (1.0 / cc) ** float(alpha); sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True)

class MSDHead(nn.Module):
    def __init__(self, h, n, dp=0.25, inner=384, nm=4):
        super().__init__(); inner = min(inner, h)
        self.pre = nn.Sequential(nn.Linear(h, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dp) for _ in range(max(1, nm))])
        self.out = nn.Linear(inner, n)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n, dp=0.25, inner=384, nm=4):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.head = MSDHead(h, n, dp, inner, nm)
    def _pool(self, o, m):
        hs = o.last_hidden_state
        return 0.5 * hs[:, 0] + 0.5 * ((hs * m.unsqueeze(-1).float()).sum(1) / m.sum(1, keepdim=True).float().clamp(1))
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        return self.head(self._pool(self.encoder(**kw), mask))

def uparams(model, e, h, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, e), (model.head, h)]:
        d, nde = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (nde if any(x in n for x in nd) else d).append(p)
        g += [{"params": d, "lr": lr, "weight_decay": wd}, {"params": nde, "lr": lr, "weight_decay": 0.0}]
    return g

class FGM:
    def __init__(self, m, eps=1.0, key="word_embeddings"): self.m, self.e, self.k, self.b = m, eps, key, {}
    def attack(self):
        for n, p in self.m.named_parameters():
            if p.requires_grad and self.k in n and p.grad is not None:
                self.b[n] = p.data.clone(); nm = torch.norm(p.grad)
                if nm != 0 and not torch.isnan(nm): p.data.add_(self.e * p.grad / nm)
    def restore(self):
        for n, p in self.m.named_parameters():
            if n in self.b: p.data = self.b[n]
        self.b = {}

class EMA:
    def __init__(self, m, d=0.999):
        self.d = d; self.s = {n: p.detach().clone() for n, p in m.named_parameters() if p.requires_grad}; self.b = {}
    def update(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad: self.s[n].mul_(self.d).add_(p.detach(), alpha=1 - self.d)
    def apply_shadow(self, m):
        self.b = {n: p.detach().clone() for n, p in m.named_parameters() if p.requires_grad}
        for n, p in m.named_parameters():
            if p.requires_grad: p.data.copy_(self.s[n])
    def restore(self, m):
        for n, p in m.named_parameters():
            if p.requires_grad and n in self.b: p.data.copy_(self.b[n])
        self.b = {}

def rdrop_kl(lp, lq):
    pl, ql = F.log_softmax(lp, -1), F.log_softmax(lq, -1); p, q = pl.exp(), ql.exp()
    return 0.5 * ((p * (pl - ql)).sum(-1) + (q * (ql - pl)).sum(-1)).mean()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); o = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        o.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(o)

@torch.no_grad()
def evaluate(model, loader, nc):
    model.eval(); P, Y, PR = [], [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        pr = F.softmax(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), -1).cpu().numpy()
        y = b["label"].cpu().numpy(); m = y >= 0
        P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y, p, pr = np.array(Y), np.array(P), np.array(PR)
    rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(y, p)), 4),
           "mcc": round(float(matthews_corrcoef(y, p)), 4)}
    try: rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro", labels=list(range(nc)))), 4)
    except Exception: rec["auroc"] = float("nan")
    return rec

print("machinery ready")


machinery ready


In [5]:
# ── Train one encoder on a held-out config (proposed CE+FGM recipe) ──
def train_enc(model_key, model_name, seed, train_df, val_df, enc, n_cls, save):
    set_seed(seed); os.makedirs(save, exist_ok=True); cfg = CFG

    # BanglaBERT is the Bangla-script specialist: train/val only on Bangla rows.
    is_script_model = (model_key == cfg.get("banglabert_script_key","banglabert")
                       and cfg.get("banglabert_script_only", False)
                       and "script" in train_df.columns)
    if is_script_model:
        tdf = train_df[train_df["script"].astype(str).str.lower().eq("bangla")].reset_index(drop=True)
        vdf = val_df[val_df["script"].astype(str).str.lower().eq("bangla")].reset_index(drop=True)
        if len(tdf) < 50 or len(vdf) < 20:
            print(f"  skip {model_key}: insufficient Bangla rows"); return None, None, float("nan")
    else:
        tdf, vdf = train_df, val_df

    tok = AutoTokenizer.from_pretrained(model_name); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=cfg["num_workers"], pin_memory=torch.cuda.is_available())
    ds = DS(tdf, tok, cfg["max_length"], enc)
    tl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **lkw)
    vl = DataLoader(DS(vdf, tok, cfg["max_length"], enc), batch_size=cfg["eval_batch_size"], shuffle=False, **lkw)

    model = Encoder(model_name, n_cls, cfg["dropout"], cfg["head_hidden_dim"], cfg["n_msd"]).to(device)
    if (not FORCE) and os.path.exists(f"{save}/best.pt"):
        model.load_state_dict(torch.load(f"{save}/best.pt", map_location=device, weights_only=True))
        print(f"  ✓ {model_key} seed{seed}: resumed"); return model, tok, float("nan")

    crit = lambda lg, t: F.cross_entropy(lg, t, label_smoothing=cfg["label_smoothing"])
    opt = torch.optim.AdamW(uparams(model, cfg["encoder_lr"], cfg["head_lr"], cfg["weight_decay"]))
    ns = math.ceil(len(tl) / cfg["grad_accum_steps"]) * cfg["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(ns * cfg["warmup_ratio"]), ns)
    scaler = torch.amp.GradScaler("cuda") if (cfg["use_fp16"] and device.type == "cuda") else None
    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None

    @torch.no_grad()
    def vmacro():
        model.eval(); P, Y = [], []
        for b in vl:
            b = {k: v.to(device) for k, v in b.items()}
            P.extend(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).argmax(-1).cpu().numpy())
            Y.extend(b["label"].cpu().numpy())
        Y = np.array(Y); m = Y >= 0
        return f1_score(Y[m], np.array(P)[m], average="macro", zero_division=0)

    best, noimp = -1.0, 0
    for ep in range(cfg["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True)
        for step, b in enumerate(tl, 1):
            b = {k: v.to(device) for k, v in b.items()}; y = b["label"]
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                l1 = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                loss = crit(l1, y) / cfg["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if fgm is not None:
                fgm.attack()
                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    la = crit(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")), y) / cfg["grad_accum_steps"]
                (scaler.scale(la) if scaler else la).backward(); fgm.restore()
            if step % cfg["grad_accum_steps"] == 0 or step == len(tl):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
        vm = vmacro()
        if vm > best:
            best, noimp = vm, 0; torch.save(model.state_dict(), f"{save}/best.pt")
        else:
            noimp += 1
        if noimp >= cfg["patience"]: break

    model.load_state_dict(torch.load(f"{save}/best.pt", map_location=device, weights_only=True))
    return model, tok, best

print("train_enc ready")


train_enc ready


In [ ]:
# ── Run robustness: 3 Bangla holdouts × 4 encoders × 3 seeds, script-aware fusion ──
def metrics(yt, yp, pr, nc):
    rec = {"macro_f1": round(float(f1_score(yt, yp, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(yt, yp, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(yt, yp)), 4),
           "mcc": round(float(matthews_corrcoef(yt, yp)), 4)}
    try: rec["macro_auroc"] = round(float(roc_auc_score(yt, pr, multi_class="ovr", average="macro", labels=list(range(nc)))), 4)
    except Exception: rec["macro_auroc"] = float("nan")
    return rec

BB_KEY = CFG.get("banglabert_script_key","banglabert"); BB_ISO = CFG.get("banglabert_script_only", False)

def masked_ens(d, act, w, names):
    num = den = None
    for i, n in enumerate(names):
        a = act[n].unsqueeze(1)
        t = w[i] * d[n] * a; num = t if num is None else num + t
        dd = w[i] * a;        den = dd if den is None else den + dd
    return num / (den + 1e-12)

def opt_w(d, act, y, names, k):
    best, bw = 1.0, np.ones(k) / k
    for _ in range(30):
        r = minimize(lambda rw: -f1_score(y, masked_ens(d, act, np.abs(rw) + 1e-9, names).argmax(-1).numpy(),
                                          average="macro", zero_division=0),
                     np.random.dirichlet(np.ones(k)), method="Nelder-Mead",
                     options={"maxiter":1500,"xatol":1e-5})
        if r.fun < best: best, bw = r.fun, np.abs(r.x) + 1e-9
    return bw / bw.sum()

per_seed_rows = []      # one row per (holdout, seed) ensemble result
for tag in configs:
    tr, va, te = HOLDOUT_SPLITS[tag]     # built in cell 2 from the master dataset; banth already excluded
    # re-verify at point of use (belt-and-suspenders — cheap, and this is the property that matters most)
    seen = set(tr["uid"]) | set(va["uid"]); leaked = len(seen & set(te["uid"]))
    assert leaked == 0, f"LEAK in {tag}: {leaked}"
    assert "banth" not in set(tr["source"]) | set(va["source"]), f"banth present in training pool for {tag}"
    classes = sorted(pd.concat([tr[LABEL], va[LABEL], te[LABEL]]).unique())
    enc = {c: i for i, c in enumerate(classes)}; nc = len(classes)
    if DEBUG:
        tr = pd.concat([g.sample(min(len(g), DEBUG_SAMPLES // nc), random_state=42) for _, g in tr.groupby(LABEL)]).reset_index(drop=True)
        va = va.sample(min(len(va), 300), random_state=42); te = te.sample(min(len(te), 300), random_state=42)
    held = tag.replace("source_holdout_", "")
    print(f"\n=== {tag} | classes={nc} | train={len(tr):,} val={len(va):,} test={len(te):,} "
          f"| train sources={sorted(tr['source'].unique())} | leak=0 OK ===")

    has_script = ("script" in va.columns) and ("script" in te.columns)
    va_bn = va["script"].astype(str).str.lower().eq("bangla").values if has_script else np.ones(len(va), bool)
    te_bn = te["script"].astype(str).str.lower().eq("bangla").values if has_script else np.ones(len(te), bool)
    y_val = va[LABEL].map(enc).values; y_test = te[LABEL].map(enc).values

    for sd in SEEDS:
        val_lg, test_lg, act_val, act_test = {}, {}, {}, {}
        for mk in RUN_MODELS:
            save = f"{OUT}/{tag}/{mk}_seed{sd}"
            model, tok, bv = train_enc(mk, CFG["models"][mk], sd, tr, va, enc, nc, save)
            if model is None:
                print(f"  ⏭ {mk} seed{sd}: skipped"); continue
            coll = Collator(tok); lkw = dict(collate_fn=coll, num_workers=0)
            vall = DataLoader(DS(va, tok, CFG["max_length"], enc), batch_size=CFG["eval_batch_size"], shuffle=False, **lkw)
            tel  = DataLoader(DS(te, tok, CFG["max_length"], enc), batch_size=CFG["eval_batch_size"], shuffle=False, **lkw)
            vlog = get_logits(model, vall).float(); tlog = get_logits(model, tel).float()
            name = f"{mk}{sd}"
            if mk == BB_KEY and BB_ISO and has_script:
                vm = torch.tensor(va_bn); tm = torch.tensor(te_bn)
                vlog[~vm] = 0.0; tlog[~tm] = 0.0
                act_val[name] = vm.float(); act_test[name] = tm.float()
            else:
                act_val[name] = torch.ones(len(va)); act_test[name] = torch.ones(len(te))
            val_lg[name] = vlog; test_lg[name] = tlog
            del model; torch.cuda.empty_cache()
        names = list(val_lg.keys())
        if not names:
            print(f"  ⏭ {tag} seed{sd}: no models"); continue
        W = opt_w(val_lg, act_val, y_val, names, len(names))
        ens_t = masked_ens(test_lg, act_test, W, names)
        pr = F.softmax(ens_t, -1).numpy(); yp = ens_t.argmax(-1).numpy()
        m = metrics(y_test, yp, pr, nc); m.update({"config": tag, "held_out": held, "seed": sd, "n_test": int(len(y_test))})
        per_seed_rows.append(m)
        print(f"  → seed{sd} ENSEMBLE on unseen {held}: macroF1={m['macro_f1']} wF1={m['weighted_f1']} acc={m['accuracy']}")

pd.DataFrame(per_seed_rows).to_csv(f"{OUT}/robustness_perseed.csv", index=False)
print(f"\n✅ saved {OUT}/robustness_perseed.csv  ({len(per_seed_rows)} rows)")



=== source_holdout_facebook_44001 | classes=5 | train=12,172 val=1,739 test=43,078 | train sources=['bd_shs', 'multilabel_12557'] | leak=0 OK ===


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglishbert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ── Aggregate over seeds and write paper_q1 robustness assets ──
ps = pd.read_csv(f"{OUT}/robustness_perseed.csv")

IN_DOMAIN_MACRO = 0.8225   # proposed model in-domain reference (from the main test split)

# pretty source names for the paper
PRETTY = {"facebook_44001":"Facebook-44K", "multilabel_12557":"Multilabel-12.5K", "bd_shs":"BD-SHS"}
order  = ["facebook_44001", "multilabel_12557", "bd_shs"]

agg = (ps.groupby("held_out", sort=False)
         .agg(n_test=("n_test","first"),
              macro_f1_mean=("macro_f1","mean"), macro_f1_std=("macro_f1","std"),
              weighted_f1_mean=("weighted_f1","mean"),
              accuracy_mean=("accuracy","mean"), mcc_mean=("mcc","mean"),
              auroc_mean=("macro_auroc","mean"))
         .reset_index())
agg["__o"] = agg["held_out"].map({k:i for i,k in enumerate(order)})
agg = agg.sort_values("__o").drop(columns="__o").reset_index(drop=True)
for c in ["macro_f1_mean","macro_f1_std","weighted_f1_mean","accuracy_mean","mcc_mean","auroc_mean"]:
    agg[c] = agg[c].round(4)
agg.to_csv(f"{OUT}/robustness_summary.csv", index=False)

# ---- LaTeX table (mean ± std on Macro-F1) ----
tab = pd.DataFrame({
    "Held-out source": agg["held_out"].map(PRETTY),
    "$n_{\\mathrm{test}}$": agg["n_test"].map(lambda v: f"{int(v):,}"),
    "Macro-F1": [f"{m:.4f} $\\pm$ {s:.4f}" if not np.isnan(s) else f"{m:.4f}"
                 for m, s in zip(agg["macro_f1_mean"], agg["macro_f1_std"])],
    "Weighted-F1": agg["weighted_f1_mean"].map(lambda v: f"{v:.4f}"),
    "Accuracy": agg["accuracy_mean"].map(lambda v: f"{v:.4f}"),
    "MCC": agg["mcc_mean"].map(lambda v: f"{v:.4f}"),
    "AUROC": agg["auroc_mean"].map(lambda v: f"{v:.4f}"),
})
tab.to_csv(f"{TAB}/table5_robustness.csv", index=False)
cols = list(tab.columns); colfmt = "llccccc"
lines = [r"\begin{table}[!htbp]", r"\centering",
         r"\caption{Source-held-out robustness of the proposed model, three Bangla-script sources, "
         r"four encoders, three seeds per configuration (mean $\pm$ std on Macro-F1). Each row trains on "
         r"the other two Bangla-script sources and tests on the held-out one. In-domain reference "
         r"Macro-F1 $= 0.8225$.}", r"\label{tab:robustness}", r"\small",
         f"\\begin{{tabular}}{{@{{}}{colfmt}@{{}}}}", r"\toprule",
         " & ".join(f"\\textbf{{{c}}}" for c in cols) + r" \\", r"\midrule"]
for _, r in tab.iterrows():
    lines.append(" & ".join(str(r[c]) for c in cols) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
open(f"{TAB}/table5_robustness.tex", "w").write("\n".join(lines))

# ---- Figure: horizontal bars with std error bars vs in-domain line ----
fig, ax = plt.subplots(figsize=(7.6, 4.0))
ylabels = [PRETTY[h] for h in agg["held_out"]]
y = np.arange(len(agg))[::-1]
means = agg["macro_f1_mean"].values; stds = agg["macro_f1_std"].fillna(0).values
ax.barh(y, means, xerr=stds, color=COL["blue"], height=0.55,
        error_kw=dict(ecolor=COL["text"], elinewidth=1.3, capsize=4))
for yi, m, s, n in zip(y, means, stds, agg["n_test"]):
    ax.text(m + s + 0.008, yi, f"{m:.3f}  (n={int(n):,})", va="center", fontsize=9.5, color=COL["text"])
ax.axvline(IN_DOMAIN_MACRO, ls="--", lw=1.3, color=COL["grey"])
ax.text(IN_DOMAIN_MACRO, len(agg)-0.4, f" in-domain = {IN_DOMAIN_MACRO:.3f}",
        color=COL["muted"], fontsize=9, va="top")
ax.set_yticks(y); ax.set_yticklabels(ylabels)
ax.set_xlabel("Macro-F1"); ax.set_xlim(0.4, 0.88)
ax.set_title("Source-held-out robustness of the proposed model", fontweight="bold", loc="left")
ax.text(0, 1.03, "three Bangla-script sources · 4 encoders · 3 seeds (mean ± std)",
        transform=ax.transAxes, fontsize=9.5, color=COL["muted"])
for ext in ("png","pdf"):
    plt.savefig(f"{FIG}/fig08_robustness.{ext}", bbox_inches="tight", dpi=200)
plt.close()

print("robustness summary (mean ± std):")
print(agg.to_string(index=False))
print(f"\n✅ wrote {TAB}/table5_robustness.(csv|tex) and {FIG}/fig08_robustness.(png|pdf)")
print(f"   mean Macro-F1 across held-outs = {agg['macro_f1_mean'].mean():.4f}")
